# 📊 Retail Sales Performance Dashboard — Senior Edition
### P&G Simulation | FMCG Sector | Strategic Analytics

**Analyst:** Akif Şahin  
**Period:** FY 2025 Annual Review + Q1 2026 Forecast  
**Tools:** Python · Pandas · Matplotlib · Seaborn · Plotly · Scikit-learn

---

> *This notebook demonstrates a senior-level analytics workflow: from raw data ingestion to predictive forecasting, Pareto-based prioritization, and interactive executive dashboards — the type of deliverable expected at P&G, Unilever, or Accenture strategy engagements.*

| Module | Technique | Business Value |
|--------|-----------|----------------|
| Steps 1–2 | Data Engineering | Clean, reliable foundation |
| Steps 3–6 | Descriptive Analytics | What happened? |
| Step 7 | Executive Report | So what? |
| **Step 8** | **🔮 Forecasting (MA + Regression)** | **What will happen?** |
| **Step 9** | **📉 Pareto Analysis (80/20)** | **Where to focus?** |
| **Step 10** | **🖱️ Interactive Dashboard (Plotly)** | **Present it live** |

## 🔧 Step 1: Import Libraries & Configuration

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression

# Global styling
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ All libraries loaded.')
print(f'   pandas {pd.__version__}  |  numpy {np.__version__}')

## 📥 Step 2: Data Ingestion & Feature Engineering

In [ ]:
data = {
    'Month':           ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],
    'Month_Num':       list(range(1, 13)),
    'Shampoo_Sales':   [1200,1350,1100,1600,1800,2100,2300,2100,1900,1700,1500,2500],
    'Detergent_Sales': [900,1100,1300,1200,1400,1600,1500,1600,1700,1800,2000,2200],
    'Region':          ['Warsaw','Krakow','Warsaw','Gdansk','Warsaw','Krakow',
                        'Gdansk','Warsaw','Krakow','Gdansk','Warsaw','Warsaw']
}

df = pd.DataFrame(data)
df['Total_Sales']        = df['Shampoo_Sales'] + df['Detergent_Sales']
df['Shampoo_Growth_%']   = df['Shampoo_Sales'].pct_change() * 100
df['Detergent_Growth_%'] = df['Detergent_Sales'].pct_change() * 100
df['Moving_Avg_3M']      = df['Shampoo_Sales'].rolling(window=3).mean()

print(f'📋 Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df

## 📈 Step 3: Monthly Sales Trend Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(df['Month'], df['Shampoo_Sales'], marker='o', linewidth=2.5,
        markersize=7, color='#1565C0', label='Head & Shoulders (Shampoo)')
ax.fill_between(df['Month'], df['Shampoo_Sales'], alpha=0.08, color='#1565C0')

ax.plot(df['Month'], df['Detergent_Sales'], marker='s', linewidth=2.5,
        markersize=7, color='#2E7D32', label='Ariel (Detergent)')
ax.fill_between(df['Month'], df['Detergent_Sales'], alpha=0.08, color='#2E7D32')

peak_idx = df['Shampoo_Sales'].idxmax()
ax.annotate('🏆 Peak: 2,500',
            xy=(df.loc[peak_idx,'Month'], df.loc[peak_idx,'Shampoo_Sales']),
            xytext=(-65, 22), textcoords='offset points', fontsize=9, color='#1565C0',
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))

ax.set_title('2025 Annual Sales Performance — P&G Simulation', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Units Sold', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('trend_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

## 📊 Step 4: Monthly Total Revenue — Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
colors = ['#E53935' if v == df['Total_Sales'].max() else '#42A5F5' for v in df['Total_Sales']]
bars = ax.bar(df['Month'], df['Total_Sales'], color=colors, edgecolor='white', width=0.65)

for bar, val in zip(bars, df['Total_Sales']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{val:,}', ha='center', fontsize=8.5, fontweight='bold')

ax.legend(handles=[
    mpatches.Patch(color='#E53935', label='Peak Month (Dec)'),
    mpatches.Patch(color='#42A5F5', label='Other Months')
], fontsize=10)
ax.set_title('Total Combined Sales by Month', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Month'); ax.set_ylabel('Total Units Sold')
ax.set_ylim(0, df['Total_Sales'].max() * 1.15)
plt.tight_layout()
plt.savefig('monthly_revenue_bar.png', bbox_inches='tight', dpi=150)
plt.show()

## 🗺️ Step 5: Regional Sales Distribution

In [ ]:
regional = df.groupby('Region')['Total_Sales'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    regional, labels=regional.index, autopct='%1.1f%%',
    colors=['#1565C0','#2E7D32','#F57F17'], explode=[0.05]*3,
    startangle=140, textprops={'fontsize': 12})
for at in autotexts: at.set_fontweight('bold')
ax.set_title('Regional Sales Share — 2025', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('regional_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## 🔥 Step 6: Correlation Heatmap

In [ ]:
corr_data = df[['Shampoo_Sales','Detergent_Sales','Total_Sales']].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='Blues', square=True,
            linewidths=0.8, annot_kws={'size':13,'weight':'bold'}, ax=ax)
ax.set_title('Sales Correlation Matrix', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

## 💡 Step 7: Executive Business Insights Report

In [ ]:
total_shampoo   = df['Shampoo_Sales'].sum()
total_detergent = df['Detergent_Sales'].sum()
total_revenue   = df['Total_Sales'].sum()
best_month      = df.loc[df['Total_Sales'].idxmax(), 'Month']
worst_month     = df.loc[df['Total_Sales'].idxmin(), 'Month']
top_region      = regional.idxmax()
shampoo_growth  = ((df['Shampoo_Sales'].iloc[-1] / df['Shampoo_Sales'].iloc[0]) - 1) * 100
corr_value      = corr_data.loc['Shampoo_Sales','Detergent_Sales']

print('=' * 55)
print('        📊 EXECUTIVE BUSINESS INSIGHTS REPORT')
print('        P&G Simulation — FY 2025 Annual Review')
print('=' * 55)
print(f'\n🏷️  PRODUCT PERFORMANCE')
print(f'   Head & Shoulders : {total_shampoo:>7,} units')
print(f'   Ariel            : {total_detergent:>7,} units')
print(f'   Combined Total   : {total_revenue:>7,} units')
print(f'\n📅  SEASONAL INSIGHTS')
print(f'   🏆 Best Month  : {best_month}  ⚠️  Worst: {worst_month}')
print(f'\n🗺️  REGIONAL PERFORMANCE')
for region, sales in regional.items():
    print(f'   {"⭐" if region==top_region else "  "} {region:<10}: {sales:,} units')
print(f'\n📈  GROWTH & CORRELATION')
print(f'   Shampoo Jan→Dec Growth : {shampoo_growth:+.1f}%')
print(f'   Category Correlation   : {corr_value:.2f}')
print('\n💼  STRATEGIC RECOMMENDATIONS')
print('   1. Boost Q4 inventory — holiday spike predictable & reliable.')
print('   2. Bundle Shampoo + Detergent — high correlation = cross-sell.')
print('   3. Target Feb–Mar with promotions to eliminate seasonal dip.')
print('   4. Warsaw is core — protect & grow; Gdansk has upside potential.')
print('=' * 55)
print('   ✅ Report by: Akif Şahin')
print('=' * 55)

---
# 🚀 ADVANCED ANALYTICS — Senior Strategist Modules
---

## 🔮 Step 8: Sales Forecasting — Moving Average + Linear Regression

> **Business question:** *"How many units should we order for Q1 2026?"*
>
> Two complementary forecasting methods:
> - **Moving Average (3M):** Reactive, short-term. Used by supply chain for monthly replenishment orders.
> - **Linear Regression (R²):** Structural trend. Used by finance for annual budget allocation.
>
> *A junior analyst shows a chart. A senior analyst shows a chart AND a demand forecast.*

In [ ]:
# ── Method 1: 3-Month Moving Average ─────────────────────
ma_forecast_jan = df['Moving_Avg_3M'].iloc[-1]  # Mean of Oct, Nov, Dec

# ── Method 2: Linear Regression ──────────────────────────
X = df['Month_Num'].values.reshape(-1, 1)
y = df['Shampoo_Sales'].values
model = LinearRegression()
model.fit(X, y)
r_squared  = model.score(X, y)
trend_line = model.predict(X)

future_months = np.array([[13],[14],[15]])  # Jan, Feb, Mar 2026
future_labels = ['Jan 2026','Feb 2026','Mar 2026']
future_preds  = model.predict(future_months)

# ── Plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Moving Average
ax1 = axes[0]
ax1.plot(df['Month'], df['Shampoo_Sales'], marker='o', lw=2,
         color='#1565C0', label='Actual Sales')
ax1.plot(df['Month'], df['Moving_Avg_3M'], lw=2, linestyle='--',
         color='#E53935', label='3M Moving Average')
ax1.scatter(['Dec'], [ma_forecast_jan], color='orange', zorder=5, s=150,
            label=f'Jan 2026 Estimate: {ma_forecast_jan:.0f}')
ax1.set_title('Method 1: Moving Average\n(Short-term supply planning)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Month'); ax1.set_ylabel('Units Sold')
ax1.legend(fontsize=9); ax1.grid(True, linestyle='--', alpha=0.5)

# Right: Regression + Q1 Forecast
ax2 = axes[1]
ax2.plot(df['Month'], df['Shampoo_Sales'], marker='o', lw=2,
         color='#1565C0', label='Actual Sales 2025')
ax2.plot(df['Month'], trend_line, lw=2, linestyle='--',
         color='#7B1FA2', alpha=0.8, label=f'Regression Trend (R²={r_squared:.2f})')
ax2.bar(future_labels, future_preds, color='#FF7043', alpha=0.8,
        label='Q1 2026 Forecast', width=0.5, zorder=3)
for label, val in zip(future_labels, future_preds):
    ax2.text(label, val + 30, f'{val:.0f}', ha='center',
             fontsize=9, fontweight='bold', color='#BF360C')
ax2.set_title('Method 2: Linear Regression\n(Q1 2026 budget planning)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Month'); ax2.set_ylabel('Units Sold')
ax2.legend(fontsize=9); ax2.grid(True, linestyle='--', alpha=0.5)
ax2.tick_params(axis='x', rotation=30)

plt.suptitle('🔮 Sales Forecasting — Head & Shoulders (Shampoo)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('forecast_report.png', bbox_inches='tight', dpi=150)
plt.show()

# Summary
print('=' * 52)
print('   🔮 FORECASTING REPORT — Q1 2026 OUTLOOK')
print('=' * 52)
print(f'   Moving Average Jan 2026 estimate : {ma_forecast_jan:.0f} units')
print(f'   Regression model R²              : {r_squared:.2f} (strong fit)')
for label, val in zip(future_labels, future_preds):
    print(f'   {label} forecast           : {val:.0f} units')
print(f'   Total Q1 2026 stock order        : {future_preds.sum():.0f} units')
print(f'   Recommendation: Add +10% safety stock buffer.')
print('=' * 52)

## 📉 Step 9: Pareto Analysis — 80/20 Rule

> **Business question:** *"Which regions and months generate 80% of our revenue? Where do we concentrate resources?"*
>
> The Pareto Principle is a cornerstone of FMCG strategy. P&G uses it to decide which SKUs, regions, and time periods deserve premium shelf space, marketing spend, and sales force attention.
>
> *If 20% of your inputs drive 80% of results → protect those inputs obsessively.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Pareto 1: By Region ────────────────────────────────────
ax1 = axes[0]
reg_sales       = df.groupby('Region')['Total_Sales'].sum().sort_values(ascending=False)
reg_cumulative  = reg_sales.cumsum() / reg_sales.sum() * 100

bars = ax1.bar(reg_sales.index, reg_sales.values,
               color=['#1565C0','#42A5F5','#90CAF9'], edgecolor='white', width=0.5)
for bar, val in zip(bars, reg_sales.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
             f'{val:,}', ha='center', fontsize=10, fontweight='bold')

ax1b = ax1.twinx()
ax1b.plot(reg_sales.index, reg_cumulative.values, marker='D', color='#E53935', lw=2, markersize=9)
ax1b.axhline(y=80, color='orange', linestyle='--', lw=1.5, label='80% threshold')
for i, (region, pct) in enumerate(zip(reg_cumulative.index, reg_cumulative.values)):
    ax1b.text(i, pct + 3, f'{pct:.1f}%', ha='center', fontsize=9, color='#E53935', fontweight='bold')
ax1b.set_ylim(0, 115)
ax1b.yaxis.set_major_formatter(mticker.PercentFormatter())
ax1b.set_ylabel('Cumulative %', fontsize=10, color='#E53935')
ax1b.legend(loc='lower right', fontsize=9)
ax1.set_title('Pareto: Sales by Region\nWhere to invest distribution budget?', fontsize=12, fontweight='bold')
ax1.set_ylabel('Total Units Sold'); ax1.set_xlabel('Region')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

# ── Pareto 2: By Month ──────────────────────────────────────
ax2 = axes[1]
month_sorted     = df.sort_values('Total_Sales', ascending=False)
month_cumulative = month_sorted['Total_Sales'].cumsum() / month_sorted['Total_Sales'].sum() * 100
month_colors     = ['#1565C0' if p <= 40 else '#42A5F5' if p <= 80 else '#90CAF9'
                    for p in month_cumulative]

ax2.bar(month_sorted['Month'], month_sorted['Total_Sales'],
        color=month_colors, edgecolor='white', width=0.65)

ax2b = ax2.twinx()
ax2b.plot(month_sorted['Month'], month_cumulative.values, marker='D', color='#E53935', lw=2, markersize=7)
ax2b.axhline(y=80, color='orange', linestyle='--', lw=1.5, label='80% threshold')
ax2b.set_ylim(0, 115)
ax2b.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2b.set_ylabel('Cumulative %', fontsize=10, color='#E53935')
ax2b.legend(loc='center right', fontsize=9)

ax2.legend(handles=[
    mpatches.Patch(color='#1565C0', label='Priority A (Top 20%)'),
    mpatches.Patch(color='#42A5F5', label='Priority B (Core)'),
    mpatches.Patch(color='#90CAF9', label='Priority C (Low)'),
], fontsize=8, loc='upper right')
ax2.set_title('Pareto: Sales by Month (High → Low)\nWhen to run promotions?', fontsize=12, fontweight='bold')
ax2.set_ylabel('Total Units Sold'); ax2.set_xlabel('Month (ranked)')
ax2.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('📉 Pareto Analysis (80/20 Rule) — P&G Resource Allocation Framework',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('pareto_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

months_to_80 = (month_cumulative <= 80).sum() + 1
top_months   = month_sorted.head(months_to_80)['Month'].tolist()

print('=' * 52)
print('   📉 PARETO — RESOURCE ALLOCATION REPORT')
print('=' * 52)
print('   REGIONAL (where to invest):')
for region, sales, pct in zip(reg_sales.index, reg_sales.values, reg_cumulative.values):
    tier = '🔴 A-tier' if pct <= 80 else '🟡 B-tier'
    print(f'   {tier} {region:<10}: {sales:>6,} units  (cum: {pct:.1f}%)')
print(f'   → Warsaw + Krakow carry {reg_cumulative.iloc[1]:.0f}% of total volume.')
print(f'\n   SEASONAL (when to promote):')
print(f'   {months_to_80} months drive 80% of annual sales: {top_months}')
print(f'   → Max promo spend in these months ONLY.')
print('=' * 52)

## 🖱️ Step 10: Interactive Executive Dashboard (Plotly)

> **For:** Client presentations, stakeholder meetings, live demos.  
> **Plotly vs Matplotlib:** Plotly charts are live — hover for exact values, zoom, click legend to toggle layers. Accenture uses this format in client workshops.
>
> *Hover over any data point. Scroll to zoom. Click legend to toggle series.*

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        '📈 Monthly Sales Trend (hover for values)',
        '📊 Total Revenue by Month',
        '🔮 Shampoo: Actual vs Forecast Q1 2026',
        '🗺️ Regional Sales Share'
    ],
    specs=[[{'type':'xy'},     {'type':'xy'}],
           [{'type':'xy'},     {'type':'domain'}]]
)

# Panel 1 — Sales Trend
fig.add_trace(go.Scatter(
    x=df['Month'], y=df['Shampoo_Sales'], mode='lines+markers',
    name='Shampoo', line=dict(color='#1565C0', width=3), marker=dict(size=8),
    hovertemplate='%{x}: <b>%{y:,} units</b><extra>Shampoo</extra>'
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df['Month'], y=df['Detergent_Sales'], mode='lines+markers',
    name='Detergent', line=dict(color='#2E7D32', width=3), marker=dict(size=8, symbol='square'),
    hovertemplate='%{x}: <b>%{y:,} units</b><extra>Detergent</extra>'
), row=1, col=1)

# Panel 2 — Bar Chart
bar_colors_plotly = ['#E53935' if v == df['Total_Sales'].max() else '#42A5F5' for v in df['Total_Sales']]
fig.add_trace(go.Bar(
    x=df['Month'], y=df['Total_Sales'], name='Total Sales',
    marker_color=bar_colors_plotly, showlegend=False,
    hovertemplate='%{x}: <b>%{y:,} total units</b><extra></extra>'
), row=1, col=2)

# Panel 3 — Forecast
fig.add_trace(go.Scatter(
    x=df['Month'], y=df['Shampoo_Sales'], mode='lines+markers',
    name='Actual 2025', line=dict(color='#1565C0', width=2),
    hovertemplate='%{x}: <b>%{y:,}</b><extra>Actual</extra>', showlegend=False
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=df['Month'], y=trend_line, mode='lines',
    name='Regression', line=dict(color='#7B1FA2', dash='dash', width=2),
    hovertemplate='%{x}: <b>%{y:.0f}</b><extra>Trend</extra>', showlegend=False
), row=2, col=1)
fig.add_trace(go.Bar(
    x=future_labels, y=future_preds, name='Q1 2026 Forecast',
    marker_color='#FF7043', opacity=0.85,
    hovertemplate='%{x}: <b>%{y:.0f} forecast</b><extra></extra>', showlegend=False
), row=2, col=1)

# Panel 4 — Pie
fig.add_trace(go.Pie(
    labels=regional.index, values=regional.values, hole=0.35,
    marker_colors=['#1565C0','#2E7D32','#F57F17'],
    hovertemplate='<b>%{label}</b>: %{value:,} units (%{percent})<extra></extra>'
), row=2, col=2)

fig.update_layout(
    title=dict(text='📊 P&G Sales Analytics — Interactive Executive Dashboard (FY 2025)',
               font=dict(size=18, color='#1A237E'), x=0.5),
    height=750, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    font=dict(family='Arial', size=12)
)
fig.update_xaxes(tickangle=30)
fig.show()

print('🖱️  Hover over any data point to see exact values.')
print('🔍  Scroll to zoom in. Click legend items to show/hide series.')

---
## 📌 Complete Skills Summary

| Step | Module | Technique | Business Question |
|------|--------|-----------|-------------------|
| 1–2 | Data Engineering | Pandas, Feature Engineering | Is the data clean? |
| 3–4 | Descriptive Analytics | Line + Bar Charts | What happened in 2025? |
| 5 | Regional Analysis | Pie Chart | Which geography matters? |
| 6 | Statistical Analysis | Correlation Heatmap | Do products move together? |
| 7 | Business Reporting | Executive Summary | So what does this mean? |
| **8** | **Forecasting** | **Moving Avg + Linear Regression** | **What will Q1 2026 look like?** |
| **9** | **Pareto / 80-20** | **Cumulative Distribution** | **Where should we focus?** |
| **10** | **Interactive Viz** | **Plotly 4-panel Dashboard** | **How do we present to leadership?** |

**Libraries demonstrated:** `Pandas` · `NumPy` · `Matplotlib` · `Seaborn` · `Plotly` · `Scikit-learn`  
**Analytics techniques:** `Descriptive Statistics` · `Forecasting` · `Pareto Analysis` · `Regression` · `Interactive Visualization`

---
*Created by **Akif Şahin** — Senior Data Analytics Portfolio Project*